In [1]:
%pip install transformers accelerate datasets peft


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#   후보 1: 디자인 전문가 페르소나
#   > "당신은 자동차 디자인 트렌드와 역사에 정통한 '자동차 디자인 전문 AI'입니다. 특히 현대자동차의 디자인 철학인
#   '센슈어스 스포티니스'와 '플루이딕 스컬프처'를 깊이 이해하고 있습니다. 사용자의 질문에 대해, 전문 지식을 바탕으로
#   시각적이고 창의적인 관점에서 상세하게 설명해주세요."

#   후보 2: 디자인 컨설턴트 페르소나
#   > "당신은 새로운 자동차 디자인 프로토타입을 기획하는 '디자인 컨설턴트'입니다. 현대차뿐만 아니라 글로벌 자동차 디자인
#   트렌드를 폭넓게 이해하고 있으며, 이를 바탕으로 사용자가 디자인 영감을 얻을 수 있도록 돕습니다. 기술적, 미학적 관점을
#   통합하여 창의적인 아이디어를 제공하듯 답변해주세요."

#   후보 3: VQA (Visual Question Answering) 어시스턴트 페르소나
#   > "당신은 텍스트 설명을 바탕으로 자동차의 이미지를 상상하고, 디자인 컨셉을 구체화하는 '디자인 시각화 AI'입니다.
#   사용자의 질문에 대해, 마치 눈앞에 자동차가 있는 것처럼 형태, 라인, 재질, 색상 등을 풍부하고 생생하게 묘사하며
#   답변해주세요."

In [2]:
# pip install -U "transformers>=4.45.0" accelerate peft datasets

import os, json, torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

# ---------------------------------------------
# 환경 설정 (권장)
# ---------------------------------------------
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cuda.matmul.allow_tf32 = True

# ---------------------------------------------
# 0) 기본 설정
# ---------------------------------------------
SYSTEM_PROMPT = (
    "당신은 자동차 디자인 트렌드와 역사에 정통한 '자동차 디자인 전문 AI'입니다. "
    "특히 현대자동차의 디자인 철학인 '센슈어스 스포티니스'와 '플루이딕 스컬프처'를 깊이 이해하고 있습니다. "
    "사용자의 질문에 대해, 제공된 문맥을 참고하여 전문 지식을 바탕으로 상세하게 설명해주세요. "
    "답변은 반드시 아래 예시와 같이 JSON 형식으로 생성해야 하며, 어떤 문맥을 참고했는지 `context number` 필드에 해당 인덱스를 '[숫자]' 형식으로 포함해야 합니다."
    "\n\n"
    "답변 예시: {\"context number\": \"[1]\", \"answer\": \"플루이딕 스컬프처는 물이나 바람이 흐르는 듯한 유기적인 선을 강조하는 디자인 철학입니다.\"}"
)

TRAIN_JSON_PATH = "./train02.jsonl"
VALID_JSON_PATH = "./validation02.jsonl"

MODEL_PATH = "kakaocorp/kanana-1.5-8b-instruct-2505"
OUTPUT_DIR = "./kanana_finetuned_model02"

MAX_LEN = None

# ---------------------------------------------
# 1) 데이터 로드/포맷
# ---------------------------------------------
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            f.seek(0)
            return [json.loads(line) for line in f]

def format_data_for_finetuning(raw_data):
    out = []
    for item in raw_data:
        question = item.get("question")
        answer = item.get("answer")
        contexts = item.get("contexts")
        positive_index = item.get("positive_index")

        if not question or not answer or not contexts or positive_index is None:
            continue
        
        combined_context = ""
        for i, context in enumerate(contexts, 1):
            combined_context += f"[{i}] {context}\n"
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "system", "content": f"다음은 참고 문맥입니다:\n{combined_context}"},
            {"role": "user", "content": question},
            {"role": "assistant", "content": f'{{"context number": "[{positive_index}]", "answer": "{answer}"}}'},
        ]
        out.append({"messages": messages})
        
    return out

train_raw = load_data(TRAIN_JSON_PATH)
train_ds  = Dataset.from_list(format_data_for_finetuning(train_raw))

valid_raw = load_data(VALID_JSON_PATH)
valid_ds  = Dataset.from_list(format_data_for_finetuning(valid_raw))

# ---------------------------------------------
# 2) 모델/토크나이저
# ---------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

if getattr(model.config, "use_cache", None):
    model.config.use_cache = False

# ---------------------------------------------
# 3) LoRA 구성
# ---------------------------------------------
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# ---------------------------------------------
# 4) 토큰화
# ---------------------------------------------
def to_prompt(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def tokenize_example(example):
    prompt = to_prompt(example["messages"])
    enc = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False,
    )
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": enc["input_ids"] # labels는 input_ids와 동일하게 설정
    }

tokenized_train_dataset = train_ds.map(tokenize_example, remove_columns=["messages"])
tokenized_valid_dataset = valid_ds.map(tokenize_example, remove_columns=["messages"])

# ---------------------------------------------
# 5) 커스텀 데이터 콜레이터
# ---------------------------------------------
def custom_data_collator(examples):
    # 각 예제의 input_ids와 attention_mask를 가져옴
    input_ids = [ex['input_ids'] for ex in examples]
    attention_mask = [ex['attention_mask'] for ex in examples]
    labels = [ex['labels'] for ex in examples]

    # 배치 내 최대 길이 찾기
    max_len = max(len(ids) for ids in input_ids)

    # 모든 시퀀스를 최대 길이에 맞춰 패딩
    padded_input_ids = []
    padded_attention_mask = []
    padded_labels = []

    for i in range(len(input_ids)):
        pad_len = max_len - len(input_ids[i])
        
        padded_input_ids.append(input_ids[i] + [tokenizer.pad_token_id] * pad_len)
        padded_attention_mask.append(attention_mask[i] + [0] * pad_len)
        
        # labels는 패딩 토큰 위치에 -100을 넣어 loss 계산에서 제외
        padded_labels.append(labels[i] + [-100] * pad_len)

    return {
        'input_ids': torch.tensor(padded_input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(padded_attention_mask, dtype=torch.long),
        'labels': torch.tensor(padded_labels, dtype=torch.long),
    }

# ---------------------------------------------
# 6) 훈련 설정
# ---------------------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    bf16=True,
    gradient_checkpointing=True,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
    optim="adamw_torch",
)

# ---------------------------------------------
# 7) Trainer 구성
# ---------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_valid_dataset,
    tokenizer=tokenizer,
    data_collator=custom_data_collator # ✅ 커스텀 콜레이터 적용
)

print("Starting LoRA fine-tuning on A100 80GB (NO checkpoint, NO cutoff)...")
trainer.train()
print("✅ Fine-tuning complete.")

# ---------------------------------------------
# 8) 최종 저장
# ---------------------------------------------
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"📦 Saved LoRA adapter & tokenizer to {OUTPUT_DIR}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 8,072,228,864 || trainable%: 0.5196


Map:   0%|          | 0/2324 [00:00<?, ? examples/s]

Map:   0%|          | 0/291 [00:00<?, ? examples/s]

/tmp/ipykernel_1020/1306117191.py:197: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting LoRA fine-tuning on A100 80GB (NO checkpoint, NO cutoff)...


Step,Training Loss
10,1.649300
20,1.292700
30,1.215800
40,1.129400
50,1.050100
60,0.993600
70,0.923500
80,0.824500
90,0.768900
100,0.652100


✅ Fine-tuning complete.
📦 Saved LoRA adapter & tokenizer to ./kanana_finetuned_model02
